# heli-lto — smoke test in Dataiku

Tests the `foca_heli` package by importing from a managed folder, without installing anything into the code environment.

## Setup

1. Create a managed folder named `heli_lto_source` (or any name; update the constant in cell 2).
2. Upload the contents of the `foca_heli/` directory into it (the inner directory containing `__init__.py`, `formulas.py`, `lto.py`, etc.). The folder structure inside the managed folder should be:
   ```
   heli_lto_source/
       foca_heli/
           __init__.py
           airframe_mtoms.py
           cli.py
           csv_io.py
           factory.py
           formulas.py
           lto.py
           operational_profiles.py
           strategies.py
           trajectory.py
   ```
   The wrapper `foca_heli/` directory is essential — Python imports the package by that name.
3. Run the cells top to bottom. Each section is independent and prints what it found.

Requires Python ≥ 3.10. No third-party dependencies.

## 1. Locate the package and add to `sys.path`

In [ ]:
import dataiku
import sys
import os
import tempfile
import shutil

MANAGED_FOLDER_NAME = "heli_lto_source"

folder = dataiku.Folder(MANAGED_FOLDER_NAME)
info = folder.get_info()
print("Folder type:", info.get("type"))
print("Is local (filesystem):", info.get("type") == "Filesystem")

In [ ]:
# Two cases:
#   (a) local filesystem folder    -> use get_path() directly
#   (b) cloud-backed folder (S3/GCS/Azure/HDFS) -> download into a temp dir

if info.get("type") == "Filesystem":
    SOURCE_DIR = folder.get_path()
    print("Using local path directly:", SOURCE_DIR)
else:
    SOURCE_DIR = tempfile.mkdtemp(prefix="heli_lto_")
    print("Cloud-backed folder; downloading to:", SOURCE_DIR)
    paths = folder.list_paths_in_partition()
    for path in paths:
        rel = path.lstrip("/")
        dst = os.path.join(SOURCE_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        with folder.get_download_stream(path) as src, open(dst, "wb") as out:
            shutil.copyfileobj(src, out)
    print("Downloaded", len(paths), "files")

# Sanity-check structure
expected_init = os.path.join(SOURCE_DIR, "foca_heli", "__init__.py")
assert os.path.exists(expected_init), (
    f"Expected to find foca_heli/__init__.py at {expected_init}. "
    "Did you upload the inner foca_heli/ directory into the managed folder?"
)

if SOURCE_DIR not in sys.path:
    sys.path.insert(0, SOURCE_DIR)

print("sys.path[0] now:", sys.path[0])
print("Python version:", sys.version_info[:3])
assert sys.version_info >= (3, 10), "foca_heli requires Python >= 3.10"

## 2. Import the package and check public API

In [ ]:
import foca_heli
from foca_heli import (
    make_strategy, compute_lto, lto_to_dict,
    PROFILES, derive_category,
    build_departure, build_arrival, to_wkt_linestring_z, to_geojson_feature,
    read_engines_csv, write_lto_csv,
    lookup_airframe, lookup_mtom_kg, BUILT_IN_AIRFRAME_MTOMS,
)

print("Loaded foca_heli from:", foca_heli.__file__)
print("Public API symbols:", len(foca_heli.__all__))
print("Built-in airframe lookup entries:", len(BUILT_IN_AIRFRAME_MTOMS))
print("Categories:", list(PROFILES.keys()))

## 3. Validation — reproduce FOCA 2015 worked examples

If these three numbers are within ~3% of FOCA's published values, the package is computing correctly in this environment.

In [ ]:
validation_cases = [
    # (label,    category,                  max_shp, n_eng, foca_target_fuel_kg, foca_appendix)
    ("AS350B2",  "SINGLE_TURBOSHAFT",       732,     1,     24.7,                "A"),
    ("A109",     "TWIN_TURBOSHAFT_LIGHT",   550,     2,     36.5,                "B"),
    ("AS332",    "TWIN_TURBOSHAFT_HEAVY",   1820,    2,     78.4,                "C"),
]

print(f"{'Helicopter':<10} {'App':<4} {'FOCA':>8} {'Computed':>10} {'Delta':>8}")
print("-" * 44)
all_ok = True
for label, cat, shp, n, target, app in validation_cases:
    s = make_strategy(category=cat, max_shp_per_engine=shp, number_of_engines=n)
    r = compute_lto(s)
    delta_pct = (r.fuel_kg - target) / target * 100
    fuel_ok = abs(delta_pct) < 3.0
    # CO2 sanity: must equal fuel × 3160 g/kg (Jet-A) for turboshaft
    expected_co2 = r.fuel_kg * 3160.0
    co2_ok = abs(r.co2_g - expected_co2) < 1.0
    flag = "OK" if (fuel_ok and co2_ok) else "FAIL"
    print(f"{label:<10} {app:<4} {target:>6.1f}kg {r.fuel_kg:>8.2f}kg {delta_pct:>+6.1f}%  {flag}")
    all_ok &= fuel_ok and co2_ok

assert all_ok, "At least one validation case failed (fuel or CO2)"
print()
print("All three FOCA worked examples reproduced within tolerance.")
print("CO2 matches expected fuel × 3160 g/kg factor.")

## 4. Per-mode breakdown for one case

Spot-check that the GI / TO / AP split makes sense.

In [ ]:
s = make_strategy(category="TWIN_TURBOSHAFT_HEAVY", max_shp_per_engine=1820, number_of_engines=2)
r = compute_lto(s)

print(f"{'Mode':<6} {'Pwr':>5} {'Time(s)':>8} {'Fuel(kg)':>10} {'NOx(g)':>9} {'HC(g)':>9} {'CO(g)':>9} {'PM(g)':>8}")
print("-" * 70)
for m in r.modes:
    print(f"{m.mode:<6} {int(m.power_fraction*100):>4}% {m.time_s:>8.0f} {m.fuel_kg:>10.2f} {m.nox_g:>9.2f} {m.hc_g:>9.2f} {m.co_g:>9.2f} {m.pm_g:>8.3f}")
print("-" * 70)
print(f"{'TOTAL':<6} {'':>5} {'':>8} {r.fuel_kg:>10.2f} {r.nox_g:>9.2f} {r.hc_g:>9.2f} {r.co_g:>9.2f} {r.pm_g:>8.3f}")
print()
print(f"CO2 = {r.co2_g/1000:.2f} kg ({r.co2_g:.0f} g)")

## 5. Round-trip a CSV through the package

Builds a small in-memory engine list, writes a CSV, reads it back through `read_engines_csv`, and runs LTO on each row. Exercises the auto-detecting schema reader and the airframe MTOM lookup.

In [ ]:
import csv
import io

# Build a small fleet CSV in OLD schema (legacy plugin format)
csv_text = """engine_name,engine_type,max_shp_per_engine,number_of_engines
HIO-360,PISTON,190,1
ARRIEL 1D1,TURBOSHAFT,732,1
PW206C,TURBOSHAFT,550,2
MAKILA 1A1,TURBOSHAFT,1820,2
PT6C-67C,TURBOSHAFT,1100,2
"""

tmp_in = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
tmp_in.write(csv_text)
tmp_in.close()

rows = read_engines_csv(tmp_in.name)
print(f"Read {len(rows)} rows. Categories assigned:")
for r in rows:
    mtom_str = f"MTOM={r.mtom_kg:.0f}kg" if r.mtom_kg else "(no MTOM lookup)"
    print(f"  {r.engine_name:<14} -> {r.category:<22} {mtom_str}")

# Compute LTO for each row
results = []
for engine_row in rows:
    strategy = make_strategy(
        category=engine_row.category,
        max_shp_per_engine=engine_row.max_shp_per_engine,
        number_of_engines=engine_row.number_of_engines,
    )
    # write_lto_csv expects dict form, not the dataclass
    results.append((engine_row, lto_to_dict(compute_lto(strategy))))

tmp_out = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
tmp_out.close()
write_lto_csv(tmp_out.name, results)

with open(tmp_out.name) as fp:
    print("\nOutput CSV:\n")
    print(fp.read())

os.unlink(tmp_in.name)
os.unlink(tmp_out.name)

## 6. Trajectory geometry

Builds a departure trajectory and exports as WKT and GeoJSON. Useful if you'll integrate with Dataiku's geo features (charts on a map background, geo joins).

In [ ]:
import json

departure = build_departure("TWIN_TURBOSHAFT_HEAVY")
print(f"Departure points: {len(departure)}")
print("First 3 points (x_m, y_m, z_m, mode):")
for p in departure[:3]:
    print(f"  ({p.x_m:.1f}, {p.y_m:.1f}, {p.z_m:.1f}, {p.mode})")
print(f"Last point: ({departure[-1].x_m:.1f}, {departure[-1].y_m:.1f}, {departure[-1].z_m:.1f})")
print(f"Final altitude (m): {departure[-1].z_m:.0f}  (FOCA LTO ceiling = 914 m / 3000 ft AGL)")

wkt = to_wkt_linestring_z(departure)
print(f"\nWKT (first 200 chars): {wkt[:200]}...")

gj = to_geojson_feature(departure, properties={"phase": "departure", "category": "TWIN_TURBOSHAFT_HEAVY"})
print(f"\nGeoJSON: {json.dumps(gj, indent=2)[:400]}...")

## 7. Optional — read engines from a Dataiku dataset

Replace `MY_DATASET_NAME` with a real dataset in your project that has columns `engine_name`, and either (`engine_type`, `max_shp_per_engine`, `number_of_engines`) or (`helicopter_category`, `max_shp_per_engine`, `number_of_engines`).

Skips silently if the dataset doesn't exist, so the notebook can be run standalone.

In [ ]:
MY_DATASET_NAME = "helicopter_engines"  # change this

try:
    ds = dataiku.Dataset(MY_DATASET_NAME)
    df = ds.get_dataframe()
    print(f"Loaded {len(df)} rows from {MY_DATASET_NAME}")
    print(df.head())
    
    # Write to a temp CSV and read through foca_heli
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False)
    df.to_csv(tmp.name, index=False)
    tmp.close()
    
    engine_rows = read_engines_csv(tmp.name)
    print(f"\nParsed {len(engine_rows)} engines via foca_heli.read_engines_csv")
    
    os.unlink(tmp.name)
except Exception as e:
    print(f"Skipped — dataset {MY_DATASET_NAME!r} not found or read failed: {e}")

## 8. Summary

If all cells above ran without error, the package works in this Dataiku environment. Next steps:

1. Build a Python recipe in your flow. The recipe code can copy the `sys.path.insert` pattern from cell 1.
2. Or, switch to a wheel install (Option B in the chat): build `foca_heli-0.1.0-py3-none-any.whl` locally with `python -m build`, upload to Dataiku, install into your code env. Then drop the `sys.path` workaround entirely — `from foca_heli import ...` works directly.
3. Or wait for the public repo and install from git: `pip install git+https://github.com/eurocontrol-asu/heli-lto.git`.